# DelFalco's Italian Restaurant - Basic Text Processing with Spacy
- You're a consultant for DelFalco's Italian Restaurant. The owner asked you to identify whether there are any foods on their menu that diners find disappointing.


- The business owner suggested you use diner reviews from the Yelp website to determine which dishes people liked and disliked. You pulled the data from Yelp(restaurant.json). Before you get to analysis, run the code cell below for a quick look at the data you have to work with.


- The owner also gave you this list of menu items and common alternate spellings.

## Here in 1st part, no modeling/Training done, using  **PhraseMatcher** we are identifying commonly occured food and using existing review rating, we tagged that food to that rating

In [60]:
import pandas as pd

In [61]:
# Set up code checking
from learntools.core import binder
binder.bind(globals())
from learntools.nlp.ex1 import *
print('Setup Complete')

ModuleNotFoundError: No module named 'learntools'

In [62]:
# Load in the data from JSON file
data = pd.read_json('../Dataset/restaurant.json')
data.head(15)

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
100086,h8l10hiVsn4DlAv3_LlsQw,UFE_5r4ewNK3jA2pRPn2ww,r5PLDU-4mSbde5XekTXSCA,5,0,0,0,Wow! Really good Italian food with a grocery s...,2018-07-08 22:42:32
1013,vvIzf3pr8lTqE_AOsxmgaA,MAmijW4ooUzujkufYYLMeQ,r5PLDU-4mSbde5XekTXSCA,4,0,0,0,We have been trying Eggplant sandwiches all ov...,2015-04-15 04:50:56
101755,27t2Z9QXd6Pm9lKfEG_LzQ,kjeX2RXvW7RhBbD2QLd5jA,r5PLDU-4mSbde5XekTXSCA,5,7,8,6,"(Lyrics - Falco - Rock Me Amadeus)\n\nOoh, Roc...",2014-01-20 16:31:22
102164,ZB4Okiod5Yxaxx5UEEvdWg,fDBybzZAL5UDscd33HCXyA,r5PLDU-4mSbde5XekTXSCA,4,3,4,2,The omnipresent crowds here speak volumes abou...,2015-12-06 23:21:51
102256,BbJWlQRUPGFxnlZbgpdrLA,62JJoUPxKxqb6snMJxi2ng,r5PLDU-4mSbde5XekTXSCA,3,0,0,0,I had a bruschetta open face sandwich basicall...,2017-12-18 22:42:51
102840,4m0DHamo6wCa-7MCtlMgng,hZFzo6HD06gtj4J7O2ki_Q,r5PLDU-4mSbde5XekTXSCA,5,0,0,0,BEST CHEESE-STEAK IN ARIZONA. BEST MEATBALL S...,2010-11-28 20:18:19
103043,1OdRyDL_32ly3_7kj0JjeA,mM2lXZF7srcZ1FaPx8L7uw,r5PLDU-4mSbde5XekTXSCA,5,1,0,1,A total place warp to New Jersey or Long Islan...,2006-09-14 00:49:58
103079,j7zyaz7uBtKSCNTVbsehoA,6bIw0iBMzJsw8uOrR92QaQ,r5PLDU-4mSbde5XekTXSCA,5,0,0,0,Unbelievable place. You'd never guess looking ...,2015-08-31 20:44:38
103169,M1-nc02YUgTifIRh28cQAg,S9Jw00eZHVj5_0sOM_C5Rg,r5PLDU-4mSbde5XekTXSCA,5,3,0,2,Stopped in today for a late lunch. It's nice ...,2010-03-26 02:13:46
103226,Ba3Kvj76TTvoCr2IU7QPUA,RdgVjfiiIXnNgA3CwEFvDg,r5PLDU-4mSbde5XekTXSCA,5,2,0,0,So.......I love Defalco's. My bf and I are add...,2012-10-11 17:46:35


In [63]:
# The owner also gave you this list of menu items and common alternate spellings.
menu = ["Cheese Steak", "Cheesesteak", "Steak and Cheese", "Italian Combo", "Tiramisu", "Cannoli",
        "Chicken Salad", "Chicken Spinach Salad", "Meatball", "Pizza", "Pizzas", "Spaghetti",
        "Bruchetta", "Eggplant", "Italian Beef", "Purista", "Pasta", "Calzones",  "Calzone",
        "Italian Sausage", "Chicken Cutlet", "Chicken Parm", "Chicken Parmesan", "Gnocchi",
        "Chicken Pesto", "Turkey Sandwich", "Turkey Breast", "Ziti", "Portobello", "Reuben",
        "Mozzarella Caprese",  "Corned Beef", "Garlic Bread", "Pastrami", "Roast Beef",
        "Tuna Salad", "Lasagna", "Artichoke Salad", "Fettuccini Alfredo", "Chicken Parmigiana",
        "Grilled Veggie", "Grilled Veggies", "Grilled Vegetable", "Mac and Cheese", "Macaroni",  
         "Prosciutto", "Salami"]

### Step 1: Plan Your Analysis

> Solution: You could group reviews by what menu items they mention, and then calculate the average rating for reviews that mentioned each item. You can tell which foods are mentioned in reviews with low scores, so the restaurant can fix the recipe or remove those foods from the menu.

### Step 2: Find items in one review

- You'll pursue this plan of calculating average scores of the reviews mentioning each menu item.

 - As a first step, you'll write code to extract the foods mentioned in a single review.

- Since menu items are multiple tokens long, you'll use PhraseMatcher which can match series of tokens.

> Hint: You should set the attr keyword argument to 'LOWER' so matching is case insensitive. An easy way to make a list of phrase documents is to loop through each item in the menu and apply the model nlp(item). This is best done in a list comprehension. From there, you can add the patterns to the matcher with matcher.add and pass in the review document to perform the matching.

In [64]:


import spacy
from spacy.matcher import PhraseMatcher

# Only 15th record value we are checking here(Index Starts from 0)
index_of_review_to_test_on = 14
#Only text column considered
text_to_test_on = data.text.iloc[index_of_review_to_test_on]
print(text_to_test_on     )

# Load the SpaCy model
nlp = spacy.blank('en')

# Create the tokenized version of text_to_test_on
review_doc = nlp(text_to_test_on)

# Create the PhraseMatcher object. The tokenizer is the first argument. Use attr = 'LOWER' to make consistent capitalization
matcher = PhraseMatcher(nlp.vocab, attr='LOWER')

# Create a list of tokens for each item in the menu
menu_tokens_list = [nlp(item) for item in menu]

# Add the item patterns to the matcher. 
matcher.add("MENU", *menu_tokens_list)
matches = matcher(review_doc)
print("\n", matches)

I had the cheesesteak sub my husband had the chicken Parmesan sub. Both were delicious but mine tasted better. Portion size: We each ordered the small one so we could save room for dessert. The sandwich covered most of a normal sized dinner plate and we had to take dessert with us.

 [(8291075388056826051, 3, 4), (8291075388056826051, 9, 11)]


>  In tupe 1st is match id, 2nd is starting position and 3rd is end position in words

> cheesesteak, chicken Parmesan

In [65]:
#Reads each tuple in one iteration
for match in matches:
    print(f"Token number {match[1]}: {review_doc[match[1]:match[2]]}")

Token number 3: cheesesteak
Token number 9: chicken Parmesan


## Step 3: Matching on the whole dataset
Now run this matcher over the whole dataset and collect ratings for each menu item. Each review has a rating, review.stars. For each item that appears in the review text (review.text), append the review's rating to a list of ratings for that item. The lists are kept in a dictionary item_ratings.

To get the matched phrases, you can reference the PhraseMatcher documentation for the structure of each match object:

A list of (match_id, start, end) tuples, describing the matches. A match tuple describes a span doc[start:end]. The match_id is the ID of the added match pattern.

> Hint: For each review, use the nlp model to convert the text to a document. Then use the matcher from exercise 1 to extract the item matches from the review text. The matches you get from the matcher are tuples (match_id, start, end), so you can do doc[start:end] to get the text phrase for that match. To get all of the unique items in the review, create a list of all the matched phrases, and convert that into a set. Finally for each of those items, append the review's rating to item_ratings. Make sure to add the item string in lowercase.

In [66]:
# Solution:

from collections import defaultdict

# item_ratings is a dictionary of lists. If a key doesn't exist in item_ratings,
# the key is added with an empty list as the value.
item_ratings = defaultdict(list)
print(item_ratings)

for idx, review in data.iterrows():
    
    #print(idx)
    # This is complete rows value with column name and value 
          #   review_id   h8l10hiVsn4DlAv3_LlsQw
          #   user_id     UFE_5r4ewNK3jA2pRPn2ww  .......
    #print("\n", review)
    
    doc = nlp(review.text)
    
    # Using the matcher from the previous exercise. 
    # Here this gives 1st is match id, 2nd is starting position and 3rd is end position in words
    
    matches = matcher(doc)
    
    # Create a set of the items found in the review text
    # #Reads each tuple in one iteration
    found_items = set([doc[match[1]:match[2]].lower_ for match in matches])
   
    # This provides food list in that row, Ex: 1st row blank set {}, 2nd row -> {'eggplant'} {'pastrami'}
    #print(found_items)
    

    # Update item_ratings with rating for each item in found_items
    # Transform the item strings to lowercase to make it case insensitive
    #Here item in for loop will be each item and corresponding food rating/ stars given in same row
    # If we have 2 items in one row text, then here 2 time for loops will run and append same stars to both the food
    
    for item in found_items:
        item_ratings[item].append(review.stars)

item_ratings

defaultdict(<class 'list'>, {})


defaultdict(list,
            {'eggplant': [4,
              5,
              5,
              5,
              3,
              5,
              5,
              5,
              4,
              5,
              4,
              5,
              4,
              4,
              4,
              5,
              5,
              4,
              5,
              2,
              5,
              3,
              5,
              5,
              5,
              4,
              3,
              5,
              5,
              3,
              5,
              5,
              4,
              3,
              5,
              5,
              4,
              2,
              4,
              3,
              5,
              5,
              5,
              3,
              4,
              4,
              5,
              5,
              2,
              4,
              4,
              5,
              5,
              2,
              5,
              2,
              5,
 

>  Output like this
defaultdict(list,
            {'chicken parmigiana': [4,
              5,
              4,
              5,
              5,
              5,
              5,
              5,
              4,
              4,
              4,
              3,
              4,
              5,
              5,
              4,
              5],
             'eggplant': [4,
              3,
              1,
              5,
              4,
              3,
              4,
              3,
              4,
              5,
              4,
              5,
              5,
              5,
              3,
              5,
              5,
              5,
              4,
              5,
              4,
              5,
              4,
              4,
              4,
              5,
              5,
              5,
              2,
              5,
              5,
              5,
              5,
              4,
              3,
              5,
              5,
              5,
              5,
              5,
              5,
              4,
              2,
              4,
              3,
              5,
              5,
              5,
              3,
              4,
              4,
              5,
              5,
              2,
              4,
              4,
              5,
              5,
              2,
              5,
              2,
              5,
              4,
              4,
              3,
              5,
              1,
              5,
              5],

## Step 4: What's the worst reviewed item?
Using these item ratings, find the menu item with the worst average rating.

> Hint: Loop through each item in item_ratings and calculate the mean, the sum of the ratings divided by the number of ratings. This is easiest using a dictionary comprehension. Then use the sorted function to sort the dictionary keys based on the dictionary values.

In [67]:
#Solution:

# There are many ways to do this. Here is one.
# Calculate the mean ratings for each menu item as a dictionary
# Here items() provides key(item name) and values(ratings) in a dictionary
mean_ratings = {item: sum(ratings)/len(ratings) for item, ratings in item_ratings.items()}
print(mean_ratings)

{'eggplant': 4.159420289855072, 'pastrami': 4.444444444444445, 'pizza': 4.339622641509434, 'pasta': 4.407766990291262, 'meatball': 4.1796875, 'cheesesteak': 4.484536082474227, 'purista': 4.666666666666667, 'prosciutto': 4.68, 'cannoli': 4.388888888888889, 'chicken parmesan': 4.2631578947368425, 'gnocchi': 4.486486486486487, 'spaghetti': 3.888888888888889, 'garlic bread': 4.128205128205129, 'chicken parm': 4.22, 'chicken parmigiana': 4.470588235294118, 'italian combo': 4.0476190476190474, 'calzone': 4.444444444444445, 'artichoke salad': 5.0, 'italian beef': 3.92, 'macaroni': 4.0, 'chicken pesto': 4.555555555555555, 'grilled veggie': 4.5, 'mac and cheese': 4.454545454545454, 'pizzas': 4.375, 'steak and cheese': 4.888888888888889, 'chicken salad': 4.6, 'calzones': 4.542857142857143, 'salami': 4.25, 'tiramisu': 4.238095238095238, 'lasagna': 4.4576271186440675, 'italian sausage': 4.30188679245283, 'roast beef': 4.142857142857143, 'corned beef': 5.0, 'ziti': 4.380952380952381, 'portobello': 

In [68]:
sorted(mean_ratings, key=mean_ratings.get)

# Used "key=mean_ratings.get", so it sorts dict based on its value/rating ascending
# This will sort the mean rationg dictionary based on the key function assigned/returned value.
# Here key function =mean_ratings.get, [in Dict, Dict.get gives values, The get() method returns the value of the item with the specified key.] 
# This gives the average value or value in dictionary. 
# Bsased on this it does sorting in ascending order ,  'chicken cutlet': 3.4, 'turkey sandwich': 3.8 ... 'turkey breast': 5.0

['chicken cutlet',
 'turkey sandwich',
 'spaghetti',
 'italian beef',
 'macaroni',
 'tuna salad',
 'italian combo',
 'garlic bread',
 'roast beef',
 'eggplant',
 'meatball',
 'chicken parm',
 'tiramisu',
 'salami',
 'chicken parmesan',
 'italian sausage',
 'pizza',
 'pizzas',
 'ziti',
 'cannoli',
 'pasta',
 'pastrami',
 'calzone',
 'mac and cheese',
 'lasagna',
 'chicken parmigiana',
 'cheesesteak',
 'gnocchi',
 'grilled veggie',
 'portobello',
 'chicken spinach salad',
 'calzones',
 'chicken pesto',
 'chicken salad',
 'purista',
 'prosciutto',
 'reuben',
 'steak and cheese',
 'artichoke salad',
 'corned beef',
 'fettuccini alfredo',
 'turkey breast']

In [69]:
# Find the worst item, and write it as a string in worst_item. This can be multiple lines of code if you want.
worst_item = sorted(mean_ratings, key=mean_ratings.get)[0]
print(worst_item)

chicken cutlet


> chicken cutlet

## Step 5: Are counts important here?
Similar to the mean ratings, you can calculate the number of reviews for each item.

In [70]:
counts = {item: len(ratings) for item, ratings in item_ratings.items()}
print("Dict of item and its order count :\n" , counts)

# Here key=counts.get, sort the counts based on the no of reviews got for that item (Which is a value in that dict)
# Here reverse=True, sord in descending
# 'pizza': 265 pasta  206

item_counts = sorted(counts, key=counts.get, reverse=True)
print("\n List of item sorted based on count :\n" ,item_counts)

#Print item and its count,
for item in item_counts:
    # Here :>25  provides 25 letter space before item. Here :>5  provides 5 letter space before item count
    print(f"{item:>25}{counts[item]:>5}")

Dict of item and its order count :
 {'eggplant': 69, 'pastrami': 9, 'pizza': 265, 'pasta': 206, 'meatball': 128, 'cheesesteak': 97, 'purista': 63, 'prosciutto': 50, 'cannoli': 72, 'chicken parmesan': 19, 'gnocchi': 37, 'spaghetti': 36, 'garlic bread': 39, 'chicken parm': 50, 'chicken parmigiana': 17, 'italian combo': 21, 'calzone': 72, 'artichoke salad': 5, 'italian beef': 25, 'macaroni': 5, 'chicken pesto': 27, 'grilled veggie': 6, 'mac and cheese': 11, 'pizzas': 32, 'steak and cheese': 9, 'chicken salad': 5, 'calzones': 35, 'salami': 28, 'tiramisu': 21, 'lasagna': 59, 'italian sausage': 53, 'roast beef': 7, 'corned beef': 2, 'ziti': 21, 'portobello': 14, 'chicken cutlet': 10, 'chicken spinach salad': 2, 'turkey sandwich': 5, 'fettuccini alfredo': 6, 'reuben': 4, 'tuna salad': 5, 'turkey breast': 1}

 List of item sorted based on count :
 ['pizza', 'pasta', 'meatball', 'cheesesteak', 'cannoli', 'calzone', 'eggplant', 'purista', 'lasagna', 'italian sausage', 'prosciutto', 'chicken parm

> Here is code to print the 10 best and 10 worst rated items. Look at the results, and decide whether you think it's important to consider the number of reviews when interpreting scores of which items are best and worst.

In [71]:
sorted_ratings = sorted(mean_ratings, key=mean_ratings.get)  # Here it sorted ascending based on rating

print("Worst rated menu items:")
for item in sorted_ratings[:10]:  # Starting 10  its in ascending order
    print(f"{item:20} Ave rating: {mean_ratings[item]:.2f} \tcount: {counts[item]}")
    
print("\n\nBest rated menu items:")
for item in sorted_ratings[-10:]: # ending 10  its in ascending order
    print(f"{item:20} Ave rating: {mean_ratings[item]:.2f} \tcount: {counts[item]}")

Worst rated menu items:
chicken cutlet       Ave rating: 3.40 	count: 10
turkey sandwich      Ave rating: 3.80 	count: 5
spaghetti            Ave rating: 3.89 	count: 36
italian beef         Ave rating: 3.92 	count: 25
macaroni             Ave rating: 4.00 	count: 5
tuna salad           Ave rating: 4.00 	count: 5
italian combo        Ave rating: 4.05 	count: 21
garlic bread         Ave rating: 4.13 	count: 39
roast beef           Ave rating: 4.14 	count: 7
eggplant             Ave rating: 4.16 	count: 69


Best rated menu items:
chicken pesto        Ave rating: 4.56 	count: 27
chicken salad        Ave rating: 4.60 	count: 5
purista              Ave rating: 4.67 	count: 63
prosciutto           Ave rating: 4.68 	count: 50
reuben               Ave rating: 4.75 	count: 4
steak and cheese     Ave rating: 4.89 	count: 9
artichoke salad      Ave rating: 5.00 	count: 5
corned beef          Ave rating: 5.00 	count: 2
fettuccini alfredo   Ave rating: 5.00 	count: 6
turkey breast        Ave ratin

> By seeing above in best 10 , we can exclude which are less count and priortize which are rated many times with good / Bad rating to consider or stop that food

## Here in 2nd part, we are doing modeling/Training done

# Natural Language Classification
You did a great such a great job for DeFalco's restaurant in the previous exercise that the chef has hired you for a new project.

The restaurant's menu includes an email address where visitors can give feedback about their food.

The manager wants you to create a tool that automatically sends him all the negative reviews so he can fix them, while automatically sending all the positive reviews to the owner, so the manager can ask for a raise.

- You will first build a model to distinguish positive reviews from negative reviews using Yelp reviews because these reviews include a rating with each review. 
- Your data consists of the text body of each review along with the star rating. 
- Ratings with 1-2 stars count as **"negative"**" , and ratings with 4-5 stars are **""positive"**". Ratings with 3 stars are **""neutral"**" and have been dropped from the data.

Let's get started. First, run the next code cell.

In [72]:
import pandas as pd

# Set up code checking
!pip install -U -t /kaggle/working/ git+https://github.com/Kaggle/learntools.git
from learntools.core import binder
binder.bind(globals())
from learntools.nlp.ex2 import *
print("\nSetup complete")

  Cloning https://github.com/Kaggle/learntools.git to c:\users\prabh\appdata\local\temp\pip-req-build-y7upq0o5


  Running command git clone -q https://github.com/Kaggle/learntools.git 'C:\Users\prabh\AppData\Local\Temp\pip-req-build-y7upq0o5'
  ERROR: Error [WinError 2] The system cannot find the file specified while executing command git clone -q https://github.com/Kaggle/learntools.git 'C:\Users\prabh\AppData\Local\Temp\pip-req-build-y7upq0o5'
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?
You should consider upgrading via the 'c:\users\prabh\anaconda3\python.exe -m pip install --upgrade pip' command.


ModuleNotFoundError: No module named 'learntools'

## Step 1: Evaluate the Approach

> Here we dont have any label to tell, its positive or negative email

> The strength of this approach is that it allows you to distinguish positive email messages from negative emails even though you don't have historical emails that you have labeled as positive or negative.

> The weakness of this approach is that emails may be systematically different from Yelp reviews in ways that make your model less accurate. For example, customers might generally use different words or slang in emails, and the model based on Yelp reviews won't have seen these words.

> If you wanted to see how serious this issue is, you could compare word frequencies between the two sources. In practice, manually reading a few emails from each source may be enough to see if it's a serious issue.

> If you wanted to do something fancier, you could create a dataset that contains both Yelp reviews and emails and see whether a model can tell a reviews source from the text content. Ideally, you'd l

## Step 2: Review Data and Create the model¶
Moving forward with your plan, you'll need to load the data. Here's some basic code to load data and split it into a training and validation set. Run this code.

In [ ]:
def load_data(csv_file, split=0.9):
    data = pd.read_csv(csv_file)
    
    # Shuffle data
    train_data = data.sample(frac=1, random_state=7)
    print(train_data.head())
    
    texts = train_data.text.values
    # Here sentiment is column with value 0 and 1,if bool(1) --> True if bool(0) --> False
    labels = [{"POSITIVE": bool(y), "NEGATIVE": not bool(y)}
              for y in train_data.sentiment.values ]
    split = int(len(train_data) * split)
    
    #Train_Y
    train_labels = [{"cats": labels} for labels in labels[:split]]
    #TEST_Y
    val_labels = [{"cats": labels} for labels in labels[split:]]
    
    return texts[:split], train_labels, texts[split:], val_labels

train_texts, train_labels, val_texts, val_labels = load_data('../input/nlp-course/yelp_ratings.csv')

In [ ]:
train_texts[:5]

In [ ]:
train_labels[:5]
# sentiment value of 1st 5 is 1,1,1,1,0

In [ ]:
print('Texts from training data\n------')
print(train_texts[:2])
print('\nLabels from training data\n------')
print(train_labels[:2])


In [ ]:
   # Create an empty model
    nlp = spacy.blank("en")

    # Create the TextCategorizer with exclusive classes and "bow" architecture
    textcat = nlp.create_pipe(
                "textcat",
                config={
                    "exclusive_classes": True,
                    "architecture": "bow"})
    nlp.add_pipe(textcat)

    # Add NEGATIVE and POSITIVE labels to text classifier
    textcat.add_label("NEGATIVE")
    textcat.add_label("POSITIVE")

## Step 3: Train Function

- Implement a function train that updates a model with training data. Most of this is general data munging, which we've filled in for you. Just add the one line of code necessary to update your mode

In [ ]:
from spacy.util import minibatch
import random

def train(model, train_data, optimizer, batch_size=8):
        losses = {}
        random.shuffle(train_data)
        # Create the batch generator with batch size = 8
        batches = minibatch(train_data, size=batch_size)
        # Iterate through minibatches
        
        # Each batch is a list of (text, label) but we need to
        # send separate lists for texts and labels to update().
        # This is a quick way to split a list of tuples into lists
        for batch in batches:
            texts, labels = zip(*batch)
            model.update(texts, labels, sgd=optimizer, losses=losses)
        return losses

In [ ]:
# Fix seed for reproducibility
spacy.util.fix_random_seed(1)
random.seed(1)

# This may take a while to run!
optimizer = nlp.begin_training()
train_data = list(zip(train_texts, train_labels))
# Call the Func train
losses = train(nlp, train_data, optimizer)
print(losses['textcat'])

In [ ]:
#We can try this slightly trained model on some example text and look at the probabilities assigned to each label.
text = "This tea cup was full of holes. Do not recommend."
doc = nlp(text)

# Here  textcat    -  TextCategorizer    -  Doc.cats    -  Assign document labels.
print(doc.cats)
# These probabilities look reasonable. Now you should turn them into an actual prediction.

In [ ]:
text = "This tea is full of tasty."
doc = nlp(text)
print(doc.cats)

## Step 4: Making Predictions
Implement a function predict that predicts the sentiment of text examples.

First, tokenize the texts using nlp.tokenizer().
Then, pass those docs to the TextCategorizer which you can get from nlp.get_pipe().
Use the textcat.predict() method to get scores for each document, then choose the class with the highest score (probability) as the predicted class.

- 
Hint: You can use nlp.tokenizer() on each text example to tokenize the input data. To make predictions, you want to get the TextCategorizer object with nlp.get_pipe(). The use .predict on the TextCategorizer to get the scores. With the scores array, the .argmax method will return the index of the highest value. Take note of the axis argument in .argmax so you're finding the max index for each example

In [ ]:
        def predict(nlp, texts):
            # Use the tokenizer to tokenize each input text example
            docs = [nlp.tokenizer(text) for text in texts]

            # Use textcat to get the scores for each doc
            textcat = nlp.get_pipe('textcat')
            scores, _ = textcat.predict(docs)

            # From the scores, find the class with the highest score/probability
            predicted_class = scores.argmax(axis=1)

            return predicted_class

In [ ]:
texts = val_texts[34:38]  # 4 records from input test set considered here
predictions = predict(nlp, texts)

for p, t in zip(predictions, texts):
    print(f"{textcat.labels[p]}: {t} \n")

It looks like your model is working well after going through the data just once. However you need to calculate some metric for the model's performance on the hold-out validation data.

# Step 5: Evaluate The Model

Implement a function that evaluates a `TextCategorizer` model. This function `evaluate` takes a model along with texts and labels. It returns the accuracy of the model, which is the number of correct predictions divided by all predictions.

First, use the `predict` method you wrote earlier to get the predicted class for each text in `texts`. Then, find where the predicted labels match the true "gold-standard" labels and calculate the accuracy.

> Hint: Use your predict function to get the predicted classes. The labels look like {'cats': {'POSITIVE':True, 'NEGATIVE': False}}, you'll need to convert these into 1s where POSITIVE is True, and 0 where POSITIVE is False. Once you have the predictions and true classes, calculate the accuracy

In [ ]:
    def evaluate(model, texts, labels):
        # Get predictions from textcat model
        """ Returns the accuracy of a TextCategorizer model. 
    
        Arguments
        ---------
        model: ScaPy model with a TextCategorizer
        texts: Text samples, from load_data function
        labels: True labels, from load_data function
        """
        
         # Get predictions from textcat model (using your predict method)   
        predicted_class = predict(model, texts)

        # From labels, get the true class as a list of integers (POSITIVE -> 1, NEGATIVE -> 0)
        true_class = [int(each['cats']['POSITIVE']) for each in labels]

        # A boolean or int array indicating correct predictions
        correct_predictions = predicted_class == true_class

        # The accuracy, number of correct predictions divided by all predictions
        accuracy = correct_predictions.mean()

        return accuracy

In [ ]:
accuracy = evaluate(nlp, val_texts, val_labels)
print(f"Accuracy: {accuracy:.4f}")



> With the functions implemented, you can train and evaluate in a loop.

## Step 6: Keep Improving
You've built the necessary components to train a text classifier with spaCy. What could you do further to optimize the model?

> 
Solution: Answer: There are various hyperparameters to work with here. The biggest one is the TextCategorizer architecture. You used the simplest model which trains faster but likely has worse performance than the CNN and ensemble models.

# Vectorizing Language
Embeddings are both conceptually clever and practically effective.

So let's try them for the sentiment analysis model you built for the restaurant. Then you can find the most similar review in the data set given some example text. It's a task where you can easily judge for yourself how well the embeddings work.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy

# Set up code checking
from learntools.core import binder
binder.bind(globals())
from learntools.nlp.ex3 import *
print("\nSetup complete")

In [ ]:
# Load the large model to get the vectors
nlp = spacy.load('en_core_web_lg')

review_data = pd.read_csv('../input/nlp-course/yelp_ratings.csv')
review_data.head()

> Calculating 44,500 document vectors takes about 20 minutes, so we'll get only the first 100. To save time, we'll load pre-saved document vectors for the hands-on coding exercises.

In [ ]:
reviews = review_data[:100]
# We just want the vectors so we can turn off other models in the pipeline
with nlp.disable_pipes():
    vectors = np.array([nlp(review.text).vector for idx, review in reviews.iterrows()])
    
vectors.shape

> The result is a matrix of 100 rows and 300 columns.

 - Why 100 rows? Because we have 1 row for each column.

- Why 300 columns? This is the same length as word vectors. See if you can figure out why document vectors have the same length as word vectors (some knowledge of linear algebra or vector math would be needed to figure this out).

In [ ]:
# Loading all document vectors from file
vectors = np.load('../input/nlp-course/review_vectors.npy')

## 1) Training a Model on Document Vectors
Next you'll train a LinearSVC model using the document vectors. It runs pretty quick and works well in high dimensional settings like you have here.

After running the LinearSVC model, you might try experimenting with other types of models to see whether it improves your results.

> Hint: Create the LinearSVC model with the regularization parameter = 10, the random state set to 1, and dual set to False. Then fit the model with the training features and labels.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(vectors, review_data.sentiment, 
                                                    test_size=0.1, random_state=1)

# Create the LinearSVC model
model = LinearSVC(random_state=1, dual=False)
# Fit the model
model.fit(X_train, y_train)
print(f'Model test accuracy: {model.score(X_test, y_test)*100:.3f}%')

# Document Similarity
For the same tea house review, find the most similar review in the dataset using cosine similarity.

## 2) Centering the Vectors
Sometimes people center document vectors when calculating similarities. That is, they calculate the mean vector from all documents, and they subtract this from each individual document's vector. Why do you think this could help with similarity metrics?

Run the following line after you've decided your answer.

> Sometimes your documents will already be fairly similar. For example, this data set is all reviews of businesses. There will be stong similarities between the documents compared to news articles, technical manuals, and recipes. You end up with all the similarities between 0.8 and 1 and no anti-similar documents (similarity < 0). When the vectors are centered, you are comparing documents within your dataset as opposed to all possible documents.

## 3) Find the most similar review

Given an example review below, find the most similar document within the Yelp dataset using the cosine similarity.

- **Hint:** To get the correct mean vector, you'll need to set the axis keyword argument to take the mean over the rows (dimension 0). The mean vector should be the same shape as the other document vectors, a 300-element vector. From there you can iterate through each centered vector and calculate the cosine simularity with the tea house review's vector. Finally to get the index of the most similar review, the .argmax() method is useful.

In [ ]:

##Here review is a sample text, we need to find similar kind review
review = """I absolutely love this place. The 360 degree glass windows with the 
Yerba buena garden view, tea pots all around and the smell of fresh tea everywhere 
transports you to what feels like a different zen zone within the city. I know 
the price is slightly more compared to the normal American size, however the food 
is very wholesome, the tea selection is incredible and I know service can be hit 
or miss often but it was on point during our most recent visit. Definitely recommend!

I would especially recommend the butternut squash gyoza."""

def cosine_similarity(a, b):
    return np.dot(a, b)/np.sqrt(a.dot(a)*b.dot(b))

review_vec = nlp(review).vector

## Center the document vectors
# Calculate the mean for the document vectors, should have shape (300,)
vec_mean = vectors.mean(axis=0)
# Subtract the mean from the vectors
centered = vectors - vec_mean

# Calculate similarities for each document in the dataset
# Make sure to subtract the mean from the review vector
sims = = np.array([cosine_similarity(review_vec - vec_mean, vec) for vec in centered])

# Get the index for the most similar document
most_similar = sims.argmax()

In [ ]:
print(review_data.iloc[most_similar].text)

> Output Received : After purchasing my final christmas gifts at the Urban Tea Merchant in Vancouver, I was surprised to hear about Teopia at the new outdoor mall at Don Mills and Lawrence when I went back home to Toronto for Christmas.
Across from the outdoor skating rink and perfect to sit by the ledge to people watch, the location was prime for tea connesieurs... or people who are just freezing cold in need of a drinK!
Like any gourmet tea shop, there were large tins of tea leaves on the walls, and although the tea menu seemed interesting enough, you can get any specialty tea as your drink. We didn't know what to get... so the lady suggested the Goji Berries... it smelled so succulent and juicy... instantly SOLD! I got it into a tea latte and watched the tea steep while the milk was steamed, and surprisingly, with the click of a button, all the water from the tea can be instantly drained into the cup (see photo).. very fascinating!

The tea was aromatic and tasty, not over powering. The price was also very reasonable and I recommend everyone to get a taste of this place :)

> Even though there are many different sorts of businesses in our Yelp dataset, you should have found another tea shop.

## 4) Looking at similar reviews
If you look at other similar reviews, you'll see many coffee shops. Why do you think reviews for coffee are similar to the example review which mentions only tea?

> Solution: Reviews for coffee shops will also be similar to our tea house review because coffee and tea are semantically similar. Most cafes serve both coffee and tea so you'll see the terms appearing together often.